# IDO-AUM Lite: numbered Colab runner
Develop on Windows; run model checks and full experiments on Colab GPU. All substantive work lives in reusable Python modules.

**Current implementation: Steps 0-5 code implemented.** Step 4 hosted test command, integrated GPU smoke and completed-stage resume passed at 9892aee. Step 5 evaluation checks and real benchmark/training/evaluation remain pending. All action flags default False; plotting remains Step 6.

## Quick walkthrough: run only what you need
Each code cell has a **Block N** heading. Blocks 1–5 are setup, once per new/restarted Python session. Every action flag starts `False`.

| Goal | Runtime / Block 1 setting | Blocks to run | Set True |
| --- | --- | --- | --- |
| Cache the actual CIFAR dataset once without using GPU allocation | Standard runtime (accelerator None), `SESSION_MODE='data_only'` | 1, 2, 3, 5, 6 (skip 4) | `RUN_CACHE_DATA` in 6 |
| Test the Step 2 model/tracker, no dataset download | GPU, `SESSION_MODE='verify'` | 1–5, 7, 9 | `RUN_COLAB_TESTS` in 7; `RUN_SMOKE` in 9 |
| Verify Step 1 with actual CIFAR data | GPU, `SESSION_MODE='verify'`; preferably cache on CPU first | 1–5, 7 if not already passed for this source/environment, 8 | `RUN_COLAB_TESTS` if needed; `RUN_DATA_CHECK` in 8 |
| Full training **when later stages exist** | GPU, `SESSION_MODE='train'` | 1–5, 6 to restore archive, 10 then 11 once for a new experiment, 12 | Future flags `RUN_BENCHMARK`, then `FREEZE_APPROVED`, then `RUN_TRAINING` |
| Resume / final evaluation **when implemented** | GPU, `SESSION_MODE='train'`, same `RUN_ID` | 1–5, 6 if local archive missing, 12 to resume, 13 to evaluate | Future `RUN_TRAINING` / `RUN_EVALUATION`; skip 10–11 for frozen runs |

**For now:** cache with Block 6 once; verify real data with Block 8 and model/tracker with Blocks 7 and 9. The smoke uses 512 generated examples; it is not actual CIFAR training or an accuracy experiment. Block 8 prepares the actual official CIFAR-10 data, not a throwaway test download.

**Flags:** enable only the action you intend to run; leave other flags False. `ACKNOWLEDGE_GPU_CHANGE` in Block 3 is only for accepting and recording a different assigned GPU on a frozen run, never for bypassing package/version mismatches. `FREEZE_APPROVED` is only for a reviewed, completed future benchmark. Changing `SESSION_MODE` does not change Colab hardware; use Runtime > Change runtime type yourself.

**Avoid repeated work:** Drive keeps one verified archive shared across RUN_IDs and both noise levels. Each new VM copies that archive to `/content/data` and extracts locally when preparation needs it. No internet re-download is needed while the Drive archive is valid. Do not rerun tests, real-data checks or smoke for each unchanged experiment condition. Within a healthy session retry just a failed action cell after fixing its cause; rerun Block 5 if the helper changed. On a new/restarted session rerun setup. Changing CPU/GPU runtime can discard `/content`; Drive survives.

Keep one stable RUN_ID for recovery, and use a new ID for a distinct experiment. Never run concurrent writers for the same experiment or cache. Disconnect the GPU when finished. Outputs are cleared in the repository notebook.


### Block 1 — Drive and session settings


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/Avilas01/ido-aum-lite.git'
REPO_REF = 'main'  # Prefer a reviewed commit SHA; recovery uses the saved SHA.
SESSION_MODE = 'verify'  # 'data_only' (CPU cache), 'verify' (GPU checks), 'train' (future GPU runs)
if SESSION_MODE not in {'data_only', 'verify', 'train'}:
    raise ValueError('Choose data_only, verify or train.')
RUN_ID = 'run-001'  # Reuse this on restart; change for a distinct experiment.
if not RUN_ID or any(x not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for x in RUN_ID):
    raise ValueError('RUN_ID must contain only letters, digits, hyphens or underscores.')
REPO_DIR = Path('/content/ido-aum-lite')
DATA_ROOT = Path('/content/data')  # Fast disposable extraction/cache.
ARCHIVE_ROOT = Path('/content/drive/MyDrive/ido-aum-lite/cache/cifar10')  # Shared across RUN_IDs.
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/ido-aum-lite/experiments') / RUN_ID
RUNS_ROOT = EXPERIMENT_ROOT / 'runs'
RESULTS_ROOT = EXPERIMENT_ROOT / 'results'
BENCHMARK_ROOT = EXPERIMENT_ROOT / 'benchmark'
FROZEN_CONFIG = EXPERIMENT_ROOT / 'experiment_config.json'
MANIFEST_PATH = EXPERIMENT_ROOT / 'manifest.json'
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
probe = EXPERIMENT_ROOT / '.write_probe'
probe.write_text('ok', encoding='utf-8')
probe.unlink()
saved_manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else None
if FROZEN_CONFIG.exists() != MANIFEST_PATH.exists():
    raise RuntimeError('Incomplete config/manifest publication: inspect Drive artifacts before proceeding; do not overwrite them.')
TARGET_REF = saved_manifest['git_sha'] if saved_manifest else REPO_REF
print('Session mode:', SESSION_MODE)
print('Persistent CIFAR archive cache:', ARCHIVE_ROOT)
print('Persistent experiment:', EXPERIMENT_ROOT)
print('Mode:', 'resume frozen experiment' if saved_manifest else 'new experiment; benchmark and freeze required')

## Source and dependencies
Set REPO_URL before continuing. On recovery the saved commit overrides REPO_REF. Existing dirty checkouts are rejected. Drive needs space for shared warm-up/final checkpoints and full optimizer resume states; check its storage quota before starting.

The notebook uses requirements-colab.txt through the CPU-safe src.colab_dependencies helper. It checks installed distribution metadata, actual imports, the torchvision torch-version requirement, CUDA availability and a tiny compiled CUDA operator in a fresh subprocess. This does not download data or train a model.

A healthy GPU-enabled pair is preserved. Missing/broken pairs are repaired **together** using the official torch 2.14.1 + torchvision 0.29.1 CUDA 12.6 wheels. If GPU runtime selection is the problem, enable a GPU before installation instead of reinstalling packages. Non-torch packages are installed only as needed, without --upgrade.

**After changing torch/torchvision, setup intentionally stops with RestartRequired. Select Runtime > Restart session (or Restart runtime), then rerun setup with the same RUN_ID.** Do not continue with cached native libraries. The next pass keeps the repaired pair. For resumed experiments, restore recorded environment versions before proceeding if they differ. Never run requirements-colab.txt on Windows. Pair source: https://pytorch.org/get-started/previous-versions/


### Public repository: no GitHub token needed
The repository is public. The next cell clones/fetches it over HTTPS without a token or GitHub sign-in. On an existing runtime, rerun this source cell to fetch current main before running Step 1; an older local checkout may not contain src/data.py. Frozen experiments keep their recorded Git SHA. No folder is automatically deleted, and dirty checkouts are rejected.

In `data_only` mode Block 3 installs lightweight dependencies only and skips all GPU/PyTorch setup. Change the actual runtime to GPU, select `verify` and rerun Blocks 1–5 for model checks. The shared Drive archive survives the switch.


### Block 2 — Fetch project source


In [ ]:
def checkout_source():
    # Public HTTPS clone/fetch: no GitHub token, credentials or sign-in prompt.
    from urllib.parse import urlsplit

    parsed = urlsplit(REPO_URL)
    if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password:
        raise ValueError('Use a plain public https://github.com/owner/repository.git URL, without credentials.')
    if 'YOUR-USERNAME' in REPO_URL:
        raise ValueError('Set REPO_URL to your GitHub repository before setup.')
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['GCM_INTERACTIVE'] = 'never'

    def git(*args, cwd=REPO_DIR):
        result = subprocess.run(['git', *map(str, args)], cwd=cwd, env=env,
                                capture_output=True, text=True, check=False)
        if result.returncode:
            raise RuntimeError(f'Git {args[0]} failed (exit {result.returncode}): {result.stderr.strip()}. '
                               'This notebook uses anonymous public cloning; check repository visibility, URL and network access.')
        return result.stdout.strip()

    try:
        if not REPO_DIR.exists():
            git('clone', REPO_URL, REPO_DIR, cwd=REPO_DIR.parent)
        if not (REPO_DIR / '.git').is_dir():
            raise RuntimeError('REPO_DIR exists but is not a Git checkout. Inspect it or choose a different REPO_DIR; no files were deleted.')
        if git('status', '--porcelain'):
            raise RuntimeError('Checkout has local changes. Preserve/review them before selecting a commit.')
        if git('remote', 'get-url', 'origin') != REPO_URL:
            raise RuntimeError('Existing checkout has a different origin; use the intended repository.')
        git('fetch', 'origin')
        remote_ref = git('rev-parse', '--verify', f'refs/remotes/origin/{TARGET_REF}') if TARGET_REF == 'main' else TARGET_REF
        git('checkout', '--detach', remote_ref)
        sha = git('rev-parse', 'HEAD')
        if saved_manifest and sha != saved_manifest['git_sha']:
            raise RuntimeError('Code identity differs from the recorded experiment.')
        return sha
    finally:
        env.clear()

GIT_SHA = checkout_source()
print('Checked out:', GIT_SHA)
print('Step 1 data module available:', (REPO_DIR / 'src' / 'data.py').is_file())
print('Step 2 modules available:', {name: (REPO_DIR / 'src' / (name + '.py')).is_file() for name in ['model', 'statistics', 'smoke']})


### Block 3 — Dependencies and configuration


In [ ]:
# No torch imports occur until dependency setup has passed in a fresh process.
if str(REPO_DIR) in sys.path:
    sys.path.remove(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
import importlib
importlib.invalidate_caches()
if globals().get('SESSION_MODE', 'verify') == 'data_only':
    # Standard-runtime archive caching: no torch installation/import or GPU probe.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
    ENVIRONMENT = None
    print('CPU data-only setup: model tests/training are disabled. No PyTorch installation.')
else:
    from src.colab_dependencies import ensure_colab_dependencies
    dependency_status = ensure_colab_dependencies(REPO_DIR)
    import torch
    import torchvision
    import src
    ENVIRONMENT = {
        'python': platform.python_version(),
        **dependency_status['versions'],
        'cuda': dependency_status['cuda'],
        'gpu': dependency_status['gpu'],
    }
    print(json.dumps(ENVIRONMENT, indent=2))
ACKNOWLEDGE_GPU_CHANGE = False  # Set True only for a reviewed change in assigned GPU.
if saved_manifest and ENVIRONMENT is not None:
    previous = saved_manifest['environment']
    package_changes = {key: (previous.get(key), value) for key, value in ENVIRONMENT.items()
                       if key in previous and key != 'gpu' and previous.get(key) != value}
    if package_changes:
        print('Changed versions:', package_changes)
        raise RuntimeError('Restore recorded dependency/runtime versions before resuming; do not change the scientific configuration.')
    if ENVIRONMENT['gpu'] != previous['gpu']:
        if not ACKNOWLEDGE_GPU_CHANGE:
            raise RuntimeError('Colab assigned a different GPU. Set ACKNOWLEDGE_GPU_CHANGE=True to document and accept the reproducibility limitation.')
        from datetime import datetime, timezone
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
        (EXPERIMENT_ROOT / f'resume_environment_{stamp}.json').write_text(
            json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')
        print('GPU change acknowledged and recorded; bitwise equivalence is not guaranteed.')
if saved_manifest:
    digest = hashlib.sha256(FROZEN_CONFIG.read_bytes()).hexdigest()
    if digest != saved_manifest['config_sha256']:
        raise RuntimeError('Frozen configuration checksum mismatch.')
config_path = FROZEN_CONFIG if saved_manifest else REPO_DIR / 'configs/default.json'
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from config import load_config
config = load_config(config_path)
print('Configuration:', json.dumps(config, indent=2))
for label, path in [('source', REPO_DIR), ('local data cache', DATA_ROOT), ('persistent archive', ARCHIVE_ROOT), ('checkpoints/logs', RUNS_ROOT), ('results', RESULTS_ROOT)]:
    print(f'{label}: {path}')

## PyTorch and GPU check
Run after dependency setup succeeds. If setup requested a runtime restart, restart and rerun setup first. This check only prints versions and GPU availability; it downloads no data and performs no training.


### Block 4 — PyTorch and GPU diagnostic


In [ ]:
if SESSION_MODE == 'data_only':
    print('GPU diagnostic skipped in CPU data-only mode.')
else:
    import torch
    import torchvision

    print("PyTorch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))


## Module command helper
The following cells require the planned research modules. They deliberately stop if those modules do not exist. Model/data loops, scoring, saving and integrity validation must be implemented in those modules, never duplicated in this notebook.

### Block 5 — Module command helper


In [ ]:
def run_module(module, *args, timeout_seconds=None):
    module_file = REPO_DIR.joinpath(*module.split('.')).with_suffix('.py')
    if not module_file.is_file():
        raise RuntimeError(f'{module} is absent from this checkout. Rerun Source and dependencies to fetch current main. If the stage is not implemented yet, complete it first.')
    try:
        from src.commands import run_python_module
    except ModuleNotFoundError as error:
        raise RuntimeError('Streaming helper is missing from this checkout. Rerun Source and dependencies; do not mix an old pinned source revision with the new notebook.') from error
    return run_python_module(module, *args, repo_root=REPO_DIR, timeout_seconds=timeout_seconds)

NOISES_TO_RUN = [0.2, 0.4]  # Use one condition per session if needed; reuse the same RUN_ID/configuration.

def require_gpu_session():
    if SESSION_MODE == 'data_only':
        raise RuntimeError("Choose GPU runtime and SESSION_MODE='verify' (or future 'train'), then rerun Blocks 1–5.")
    if not torch.cuda.is_available():
        raise RuntimeError('GPU unavailable; select a GPU runtime and rerun setup.')


## Persistent CIFAR archive (CPU-friendly)
Run once on a standard CPU runtime before requesting a GPU for model work. This downloads the official archive containing **50,000 training + 10,000 test images**, verifies its official MD5 and saves one copy on Drive. It does not extract images, import PyTorch, generate labels or train. A valid local archive or Drive archive skips internet download. If the current VM already downloaded the archive, this backs it up rather than downloading again.

The first internet download can be slow; `CACHE_TIMEOUT_SECONDS=3600` bounds this cell. You see byte progress and heartbeat/stage timings. A failed/partial archive is never published as the final Drive cache. Interrupted downloads may need restarting; HTTP partial-download resume is not implemented. Keep the Drive cache after success. Archive corruption triggers a fresh download with verification; Drive read/write/quota errors stop clearly and retain valid local files. Allow roughly 170 MB on Drive plus local archive/extraction space.


### Block 6 — Cache / restore actual CIFAR archive


In [ ]:
RUN_CACHE_DATA = False
CACHE_TIMEOUT_SECONDS = 3600
if RUN_CACHE_DATA:
    run_module('src.archive_cache', '--data-root', DATA_ROOT, '--archive-root', ARCHIVE_ROOT,
               timeout_seconds=CACHE_TIMEOUT_SECONDS)


## Hosted tests (Steps 0–4)
Set RUN_COLAB_TESTS=True below and run the cell. It installs only test extras, then runs all tests, including model/tracker checks, Step 3 loss/gradient checks and Step 4 tiny staged-resume fixtures. Windows runs only NumPy/configuration/notebook tests; PyTorch checks run here with generated fixtures, no CIFAR download and no full-model training. Confirm this passes before running the Step 2 smoke.


### Block 7 — Hosted tests (Steps 0–4)


In [ ]:
RUN_COLAB_TESTS = False
if RUN_COLAB_TESTS:
    require_gpu_session()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-dev.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)


## Step 1 verification: observable real CIFAR-10 preparation
First run the hosted tests with RUN_COLAB_TESTS=True. Then set RUN_DATA_CHECK=True below. Both noise levels are prepared by ONE unbuffered child process, loading the official training split once from the SAME DATA_ROOT (/content/data by default). Valid extracted files/archives are reused, not downloaded again. Only one weak/strong sample per condition is transformed; this does not run the model or train anything.

You should see START/END messages and timings for torchvision import, cache checksum verification, download/archive verification (only if needed), extraction, sample selection, corrupted-label generation and each metadata file. A heartbeat identifies the current stage every 15 seconds. Download progress and errors are streamed into this cell; the parent waiting for a child is normal and not itself a diagnosis.

PREPARATION_TIMEOUT_SECONDS=900 bounds this check only, not training. If you interrupt or it times out, the child is stopped and files are preserved. On retry, compatible final metadata are verified/reused and valid partial metadata are completed; conflicting/corrupt finals fail clearly instead of being overwritten. Cache the archive with Block 6 on a standard CPU runtime first. Block 8 automatically restores a verified Drive archive before extraction; its 900-second preparation budget is intended for cached input. The initial archive may need a fresh download if interruption left it incomplete. Keep image/cache I/O under /content/data and metadata on Drive. Stage timings can be nested and are not additive.

After preparation, the cell verifies exact label-change counts, original class balance, matching image IDs and the unchanged 10,000-image test split. Clean labels/masks are used only for these sanity checks. Step 1 verification requires both the hosted tests and these checks to pass.

After loading an updated notebook, rerun source checkout, dependency setup and Module command helper before this cell. A browser refresh alone does not update the source in /content. Stop any older preparation before retrying. If you cannot confirm the old child stopped, use a fresh Colab runtime to avoid two writers; its local cache is restored from the verified Drive archive when available. Keep the same RUN_ID and Drive artifacts.


### Block 8 — Step 1 real-data verification


In [ ]:
RUN_DATA_CHECK = False
PREPARATION_TIMEOUT_SECONDS = 900
if RUN_DATA_CHECK:
    require_gpu_session()
    import numpy as np
    import importlib
    import src.data as data_module
    importlib.reload(data_module)  # Refresh an older already-imported preparation API.
    from config import configuration_fingerprint
    from src.data import load_training_metadata, load_analysis_metadata, load_cifar10_test
    from src.preparation_progress import PreparationProgress

    preparation_root = EXPERIMENT_ROOT / 'data_checks' / configuration_fingerprint(config)[:12]
    print(f'Data cache: {DATA_ROOT}; metadata: {preparation_root}', flush=True)
    run_module('src.data', '--prepare', '--config', config_path, '--noise', *config['noise_ratios'],
               '--data-root', DATA_ROOT, '--archive-root', ARCHIVE_ROOT, '--metadata-root', preparation_root,
               timeout_seconds=PREPARATION_TIMEOUT_SECONDS)

    reference_ids = None
    with PreparationProgress() as verification:
        for ratio in config['noise_ratios']:
            metadata_root = preparation_root / f'noise_{round(100*ratio)}'
            with verification.stage(f'{ratio:.0%}: verify saved data metadata'):
                training = load_training_metadata(metadata_root)
                analysis = load_analysis_metadata(metadata_root)  # Sanity checks only.
                expected = round(ratio * config['train_size'])
                assert len(training.noisy_labels) == config['train_size']
                assert int(analysis.is_corrupted.sum()) == expected
                counts = np.bincount(analysis.clean_labels, minlength=10)
                assert np.all(counts == config['train_size'] // 10)
                if reference_ids is None:
                    reference_ids = training.original_indices
                else:
                    np.testing.assert_array_equal(reference_ids, training.original_indices)
                print(f'{ratio:.0%} noise: {len(training.noisy_labels)} images, {expected} changed labels; class counts: {counts.tolist()}', flush=True)

        # Load the test split once, from the same cache; never download/corrupt it here.
        with verification.stage('Verify untouched official test split'):
            test_data = load_cifar10_test(DATA_ROOT, download=False, progress=verification)
            assert len(test_data) == config['test_size'] == 10000
            sample, label = test_data[0]
            assert tuple(sample.shape) == (3, 32, 32) and 0 <= label < 10
            print(f'Official test split: {len(test_data)} images; shape: {tuple(sample.shape)}', flush=True)
    print('Step 1 real-data checks passed. Confirm the hosted tests above also passed.', flush=True)


## Step 2 verification: GPU model and tracking smoke
After hosted tests pass, set RUN_SMOKE=True below. This runs one plain-CE epoch on 512 generated image-shaped tensors (four minibatches at the default batch size), then one deterministic full statistics pass. It checks output [batch,10], finite losses/gradients, WE/count bounds, finite AUM, train/eval mode restoration and snapshot reload. It downloads no CIFAR/ImageNet data and measures no test accuracy.

A timestamped report and epoch_001.npz snapshot are saved to Drive under smoke_step2/<Git SHA>/. This is an isolated engineering check, not a full experiment, a six-epoch warm-up or a runtime benchmark for real CIFAR-10. Step 2 hosted verification is complete when the tests pass and the command prints Step 2 smoke passed. Model/optimizer resume and the three full training branches belong to Step 4.


### Block 9 — Step 2 synthetic GPU smoke


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    require_gpu_session()
    run_module('src.smoke', '--config', config_path, '--device', 'cuda',
               '--output-root', EXPERIMENT_ROOT / 'smoke_step2' / GIT_SHA[:12])


## New experiments only: benchmark, then freeze
Skip these two cells on restart. Benchmark outputs are isolated from full results. Include both views, statistics/CE inference, loading and persistent checkpoint I/O. If necessary, select the plan's uniform fallback in central configuration and rebenchmark using `--config`. Never use test/detection accuracy for that choice.

The benchmark must save its candidate scientific settings at `benchmark/experiment_config.json`. Set `FREEZE_APPROVED=True` only after the runtime estimate fits the total project budget. No full experiment begins until configuration is frozen.

### Block 10 — Step 4 benchmark (hosted checks pending)


In [ ]:
RUN_BENCHMARK = False  # Step 4 implemented; leave False until hosted verification passes.
if RUN_BENCHMARK:
    require_gpu_session()
    if not FROZEN_CONFIG.exists():
        run_module('src.train', '--benchmark', '--config', REPO_DIR / 'configs/default.json',
                   '--noise', 0.2, '--device', 'cuda', '--data-root', DATA_ROOT,
                   '--archive-root', ARCHIVE_ROOT, '--output-root', BENCHMARK_ROOT)
    else:
        print('Existing frozen experiment: benchmark skipped.')

### Block 11 — Freeze approval (after benchmark)


In [ ]:
FREEZE_APPROVED = False  # New runs only: set True after reviewing benchmark/fallback.
if not FREEZE_APPROVED:
    print('Freeze disabled. Leave False until the future benchmark is implemented and reviewed.')
elif not FROZEN_CONFIG.exists():
    if ENVIRONMENT is None:
        raise RuntimeError('GPU setup required before freezing an experiment.')
    candidate = BENCHMARK_ROOT / 'experiment_config.json'
    raw_config = candidate.read_bytes()
    load_config(candidate)  # Reuse portable configuration validation before publication.
    manifest = {'run_id': RUN_ID, 'git_sha': GIT_SHA,
                'config_sha256': hashlib.sha256(raw_config).hexdigest(), 'environment': ENVIRONMENT}
    config_tmp = FROZEN_CONFIG.with_suffix('.json.tmp')
    manifest_tmp = MANIFEST_PATH.with_suffix('.json.tmp')
    config_tmp.write_bytes(raw_config)
    manifest_tmp.write_text(json.dumps(manifest, indent=2) + chr(10), encoding='utf-8')
    if hashlib.sha256(config_tmp.read_bytes()).hexdigest() != manifest['config_sha256']:
        raise RuntimeError('Persistent configuration write verification failed.')
    config_tmp.replace(FROZEN_CONFIG)
    manifest_tmp.replace(MANIFEST_PATH)
    print('Published frozen configuration and manifest:', EXPERIMENT_ROOT)
else:
    print('Existing configuration preserved:', FROZEN_CONFIG)


## Run or resume
These commands are identical on first execution and restart. The runner must start missing stages, restore interrupted branches and skip complete branches, rejecting config/code/checksum mismatches. It must persist full state after each completed epoch and retain the previous verified checkpoint. It may repeat unfinished epoch work; it must not duplicate completed histories/logs.

Before running, confirm the printed paths point to Drive. Full state includes model/BatchNorm, optimizer/scaler, epoch, WE/margins/counts, all RNG/DataLoader states, configuration and dataset/noisy-label checksums.

### Block 12 — Full training / resume (hosted checks pending)


In [ ]:
RUN_TRAINING = False  # Leave False until hosted smoke, resume tests and benchmark/freeze are complete.
if RUN_TRAINING:
    require_gpu_session()
    if not FROZEN_CONFIG.exists() or not MANIFEST_PATH.exists():
        raise RuntimeError('Freeze the benchmark-selected configuration and manifest before training.')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA required for full runs.')
    gpu_change_args = ['--acknowledge-gpu-change'] if ACKNOWLEDGE_GPU_CHANGE else []
    for noise in NOISES_TO_RUN:
        run_module('src.train', '--config', FROZEN_CONFIG, '--noise', noise,
                   '--device', 'cuda', '--data-root', DATA_ROOT,
                   '--archive-root', ARCHIVE_ROOT, '--output-root', RUNS_ROOT, '--resume', *gpu_change_args)

## Evaluate, plot and inspect persistent outputs
Evaluation alone owns test accuracy, detector ROC-AUC and metrics.csv. It must cache completed checkpoint identities to avoid repeated evaluation/duplicate rows. Plots use saved metrics/logs. Both stages write directly to Drive.

### Block 13 - Final evaluation / future plotting


In [ ]:
RUN_EVALUATION = False  # Step 5 implemented; run only after completed final checkpoints exist.
RUN_PLOTTING = False  # Step 6 remains unimplemented.
if RUN_EVALUATION:
    require_gpu_session()
    gpu_change_args = ['--acknowledge-gpu-change'] if ACKNOWLEDGE_GPU_CHANGE else []
    run_module('src.evaluate', '--device', 'cuda', '--data-root', DATA_ROOT,
               '--archive-root', ARCHIVE_ROOT, '--runs', RUNS_ROOT,
               '--output-root', RESULTS_ROOT, *gpu_change_args)
    import pandas as pd
    from IPython.display import display
    metrics_path = RESULTS_ROOT / 'metrics.csv'
    display(pd.read_csv(metrics_path))
    print('Persistent metrics:', metrics_path)
    print('Persistent checkpoints/logs:', RUNS_ROOT)
if RUN_PLOTTING:
    run_module('src.plots', '--results', RESULTS_ROOT, '--runs', RUNS_ROOT)
    from IPython.display import display, Image
    for figure in sorted((RESULTS_ROOT / 'figures').glob('*.png')):
        display(Image(filename=str(figure)))


## Restart / interruption checklist
1. Stop the interrupted action before retrying; never launch two writers. If the child may still be running, start a fresh runtime.
2. Open the latest GitHub notebook (a saved Drive copy does not update itself), select the runtime appropriate to your goal and reuse RUN_ID. Run Blocks 1–5; CPU archive-only sessions skip 4. The source cell fetches current main for unfrozen runs; frozen runs keep their saved SHA. A browser refresh alone does not update the checkout or Python imports. After updating code in a live session, restart Python and rerun setup to avoid stale imported helpers.
3. If dependency setup requests a restart after repairing torch/torchvision, select Runtime > Restart session and rerun Blocks 1–5. Do not reinstall a working Colab GPU stack manually.
4. Run Block 6 only when you need to create/restore the local archive. Block 8 also restores it automatically. A valid Drive archive avoids another internet download; extraction into a new VM may still be needed. Existing compatible metadata on Drive are verified/reused. Keep the cache independent of RUN_ID.
5. For the currently implemented stages, rerun only the needed verification (7, 8 and/or 9). Passing these does not mean full model training has run.
6. Future full experiments: preserve RUN_ID, frozen config/manifest, environment and Git SHA; skip 10–11, enable 12 to request `--resume`, then 13 after completion. The full checkpoint/resume/evaluation modules are not implemented yet. Do not erase Drive outputs or silently regenerate a frozen configuration; inspect incomplete publications before proceeding.

Colab resources and lifetime are limited; use a standard runtime for archive downloads and disconnect unused GPUs. See [Colab FAQ](https://research.google.com/colaboratory/faq.html).
